<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_4_RAG_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

cell 1

In [1]:
!pip install -q \
    llama-index \
    llama-index-embeddings-huggingface \
    sentence-transformers \
    sentencepiece \
    protobuf \
    fastapi \
    uvicorn \
    pyngrok \
    nest-asyncio \
    transformers \
    accelerate \
    bitsandbytes \
    pandas \
    tqdm \
    requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


cell 2

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import re
import json
import hashlib
import shutil
import gc
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple, Set

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

PROJECT_DIR = Path("/content/drive/MyDrive/llm")

# Thesis 3 outputs: guideline/document corpus only
RAG_META_DIR = PROJECT_DIR / "rag_metadata"
CHILD_NODES_PATH = RAG_META_DIR / "child_nodes.jsonl"
PARENT_LOOKUP_PATH = RAG_META_DIR / "parent_lookup.json"
DOCUMENT_INVENTORY_PATH = RAG_META_DIR / "document_inventory.csv"

# Final 50 patient-profile JSON
# The code searches these locations. Put your final JSON in one of these paths.
FINAL_PROFILE_CANDIDATES = [
    PROJECT_DIR / "selected_50_final_patient_profiles_for_rag.json",
    PROJECT_DIR / "patient_states" / "selected_50_final_patient_profiles_for_rag.json",
    PROJECT_DIR / "data_processed" / "selected_50_final_patient_profiles_for_rag.json",
]

FINAL_PATIENT_PROFILE_PATH = None
for p in FINAL_PROFILE_CANDIDATES:
    if p.exists():
        FINAL_PATIENT_PROFILE_PATH = p
        break

if FINAL_PATIENT_PROFILE_PATH is None:
    raise FileNotFoundError(
        "Could not find selected_50_final_patient_profiles_for_rag.json. "
        "Place it in one of these locations:\n"
        + "\n".join(str(p) for p in FINAL_PROFILE_CANDIDATES)
    )

# Thesis 4 outputs
RAG_INDEX_DIR = PROJECT_DIR / "rag_index"
INDEX_STORAGE_DIR = RAG_INDEX_DIR / "storage"
INDEX_MANIFEST_PATH = RAG_INDEX_DIR / "index_manifest.json"

RAG_OUTPUT_DIR = PROJECT_DIR / "rag_outputs"
RAG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAG_INDEX_DIR.mkdir(parents=True, exist_ok=True)
INDEX_STORAGE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("FINAL_PATIENT_PROFILE_PATH:", FINAL_PATIENT_PROFILE_PATH)
print("CHILD_NODES_PATH:", CHILD_NODES_PATH)
print("PARENT_LOOKUP_PATH:", PARENT_LOOKUP_PATH)
print("INDEX_STORAGE_DIR:", INDEX_STORAGE_DIR)
print("RAG_OUTPUT_DIR:", RAG_OUTPUT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/llm
FINAL_PATIENT_PROFILE_PATH: /content/drive/MyDrive/llm/patient_states/selected_50_final_patient_profiles_for_rag.json
CHILD_NODES_PATH: /content/drive/MyDrive/llm/rag_metadata/child_nodes.jsonl
PARENT_LOOKUP_PATH: /content/drive/MyDrive/llm/rag_metadata/parent_lookup.json
INDEX_STORAGE_DIR: /content/drive/MyDrive/llm/rag_index/storage
RAG_OUTPUT_DIR: /content/drive/MyDrive/llm/rag_outputs


cell 3

In [38]:
# ============================================================
# Main configuration
# ============================================================

import os

# Set True only when you want to rebuild embeddings/index.
REBUILD_INDEX = False

# Automatically rebuild if child_nodes.jsonl changed.
AUTO_REBUILD_IF_CORPUS_CHANGED = True

# Embedding model for guideline evidence chunks
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"

# For Colab T4, keep embedding/reranking on CPU to save GPU memory for Mistral.
EMBED_DEVICE = "cpu"
EMBED_BATCH_SIZE = 16

# Optional index splitting for unusually long evidence nodes
SPLIT_LONG_INDEX_NODES = True
MAX_INDEX_NODE_CHARS = 2500
TARGET_INDEX_NODE_CHARS = 1800

# Retrieval
RETRIEVE_TOP_K = 20
USE_RERANKER = True
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"
RERANK_DEVICE = "cpu"
RERANK_TOP_N = 10
RERANK_BATCH_SIZE = 8
RERANK_MAX_LENGTH = 512

FINAL_EVIDENCE_TOP_N = 8
MAX_EVIDENCE_CHARS_PER_SOURCE = 2000
INCLUDE_PARENT_CONTEXT = False
MAX_PARENT_CONTEXT_CHARS = 500

# Patient context controls
MAX_PATIENT_PROMPT_CONTEXT_CHARS = 5000
MAX_PATIENT_RETRIEVAL_CONTEXT_CHARS = 1800

# LLM
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
USE_4BIT = True
DEFAULT_MAX_NEW_TOKENS = 900

# API
RUN_FASTAPI_SERVER = True
START_NGROK = True   # True because you want frontend/ngrok access.

# ============================================================
# Tokens
# ============================================================
# Paste your ngrok token directly here.
# Do not upload/share this notebook publicly after pasting the real token.

HF_TOKEN = ""  # Optional. Paste Hugging Face token here only if needed.

NGROK_AUTH_TOKEN = "3CenhdMC9xr7r8lyEVTrEJ9wwZ5_6CE9ZR6vzT8ZSxsaV6bVd"

# ============================================================
# Configuration check
# ============================================================

print("Configuration loaded.")
print("REBUILD_INDEX:", REBUILD_INDEX)
print("AUTO_REBUILD_IF_CORPUS_CHANGED:", AUTO_REBUILD_IF_CORPUS_CHANGED)
print("EMBED_MODEL_NAME:", EMBED_MODEL_NAME)
print("EMBED_DEVICE:", EMBED_DEVICE)
print("USE_RERANKER:", USE_RERANKER)
print("RERANK_MODEL_NAME:", RERANK_MODEL_NAME)
print("RERANK_DEVICE:", RERANK_DEVICE)
print("FINAL_EVIDENCE_TOP_N:", FINAL_EVIDENCE_TOP_N)
print("MODEL_ID:", MODEL_ID)
print("USE_4BIT:", USE_4BIT)
print("DEFAULT_MAX_NEW_TOKENS:", DEFAULT_MAX_NEW_TOKENS)
print("RUN_FASTAPI_SERVER:", RUN_FASTAPI_SERVER)
print("START_NGROK:", START_NGROK)
print("HF token available:", bool(HF_TOKEN.strip()))
print("ngrok token available:", bool(NGROK_AUTH_TOKEN.strip()))

Configuration loaded.
REBUILD_INDEX: False
AUTO_REBUILD_IF_CORPUS_CHANGED: True
EMBED_MODEL_NAME: BAAI/bge-large-en-v1.5
EMBED_DEVICE: cpu
USE_RERANKER: True
RERANK_MODEL_NAME: BAAI/bge-reranker-base
RERANK_DEVICE: cpu
FINAL_EVIDENCE_TOP_N: 8
MODEL_ID: mistralai/Mistral-7B-Instruct-v0.3
USE_4BIT: True
DEFAULT_MAX_NEW_TOKENS: 900
RUN_FASTAPI_SERVER: True
START_NGROK: True
HF token available: False
ngrok token available: True


cell 4

In [5]:
def normalize_patient_id(x: Any) -> str:
    """
    Converts patient IDs into stable string format.
    Handles values like 722128, 722128.0, or '722128'.
    """
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def compute_file_hash(path: Path, block_size: int = 1024 * 1024) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            sha.update(block)
    return sha.hexdigest()


def stable_hash(text: Any, length: int = 12) -> str:
    return hashlib.sha1(str(text).encode("utf-8", errors="ignore")).hexdigest()[:length]


def re_sub_multi_space(text: str) -> str:
    text = re.sub(r"[ \t]{2,}", " ", str(text))
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def clean_text_for_prompt(text: Any) -> str:
    text = str(text)
    text = text.replace("\u00ad", "")
    text = text.replace("￾", "")
    text = text.replace("\xa0", " ")
    text = text.replace(" ", " ")
    text = "\n".join(line.rstrip() for line in text.splitlines())
    text = re_sub_multi_space(text)
    return text.strip()


def truncate_text(text: Any, max_chars: int) -> str:
    text = clean_text_for_prompt(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "\n...[truncated]"


def normalize_key(k: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(k).strip().lower()).strip("_")


def stringify_value(value: Any, indent: int = 0, max_items: int = 30) -> str:
    """
    Converts nested profile fields into readable compact text.
    """
    pad = " " * indent

    if value is None:
        return ""

    if isinstance(value, (str, int, float, bool)):
        return f"{pad}{value}"

    if isinstance(value, list):
        lines = []
        for item in value[:max_items]:
            item_text = stringify_value(item, indent=indent + 2, max_items=max_items)
            if item_text.strip():
                lines.append(f"{pad}- {item_text.strip()}")
        if len(value) > max_items:
            lines.append(f"{pad}- ... {len(value) - max_items} more items")
        return "\n".join(lines)

    if isinstance(value, dict):
        lines = []
        for k, v in value.items():
            if v is None or v == "" or v == [] or v == {}:
                continue

            if isinstance(v, (str, int, float, bool)):
                lines.append(f"{pad}- {k}: {v}")
            else:
                nested = stringify_value(v, indent=indent + 2, max_items=max_items)
                if nested.strip():
                    lines.append(f"{pad}- {k}:\n{nested}")
        return "\n".join(lines)

    return f"{pad}{str(value)}"


def get_first_existing_field(record: Dict[str, Any], aliases: List[str]) -> Any:
    """
    Direct top-level lookup with flexible key spelling.
    """
    normalized_map = {normalize_key(k): k for k in record.keys()}

    for alias in aliases:
        nk = normalize_key(alias)
        if nk in normalized_map:
            return record[normalized_map[nk]]

    return None


def is_truthy_flag(v: Any) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return v != 0
    s = str(v).strip().lower()
    if s in ["", "0", "false", "no", "none", "null", "nan", "not available", "absent"]:
        return False
    return True

cell 5

In [6]:
def extract_profile_records(raw_data: Any) -> List[Dict[str, Any]]:
    """
    Supports common JSON shapes:
    1. [profile, profile, ...]
    2. {"patients": [...]}
    3. {"profiles": [...]}
    4. {"722128": {profile}, "723327": {profile}, ...}
    """
    if isinstance(raw_data, list):
        return raw_data

    if isinstance(raw_data, dict):
        root_keys = ["patients", "profiles", "patient_profiles", "records", "data"]

        for key in root_keys:
            if key in raw_data and isinstance(raw_data[key], list):
                return raw_data[key]

        # Dictionary keyed by patient_id
        records = []
        for k, v in raw_data.items():
            if isinstance(v, dict):
                rec = dict(v)
                rec.setdefault("patient_id", k)
                records.append(rec)

        if records:
            return records

    raise ValueError("Unsupported patient-profile JSON structure.")


def collect_risk_flags(profile: Dict[str, Any]) -> List[str]:
    raw = get_first_existing_field(profile, [
        "final_risk_flags",
        "risk_flags",
        "clinical_risk_flags",
        "patient_risk_flags",
    ])

    flags = []

    if raw is None:
        return flags

    if isinstance(raw, dict):
        for k, v in raw.items():
            if is_truthy_flag(v):
                if isinstance(v, bool):
                    flags.append(str(k))
                else:
                    flags.append(f"{k}: {v}")

    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, str):
                if item.strip():
                    flags.append(item.strip())
            elif isinstance(item, dict):
                name = (
                    item.get("flag")
                    or item.get("risk_flag")
                    or item.get("name")
                    or item.get("id")
                    or item.get("trigger")
                    or item.get("trigger_id")
                )
                status = item.get("status") or item.get("value") or item.get("meaning")
                if name:
                    flags.append(str(name) if not status else f"{name}: {status}")

    elif isinstance(raw, str):
        flags.extend([x.strip() for x in re.split(r"[,\n;|]+", raw) if x.strip()])

    return sorted(set(flags))


def collect_verification_triggers(profile: Dict[str, Any]) -> List[Dict[str, Any]]:
    raw = get_first_existing_field(profile, [
        "verification_triggers",
        "patient_verification_triggers",
        "triggers",
    ])

    triggers = []

    if raw is None:
        return triggers

    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, str):
                triggers.append({
                    "trigger": item,
                    "meaning": "",
                    "expected_answer_behavior": "",
                    "validation_status": "draft heuristic; requires guideline or clinician validation",
                })
            elif isinstance(item, dict):
                trigger_name = (
                    item.get("trigger")
                    or item.get("trigger_id")
                    or item.get("id")
                    or item.get("name")
                    or item.get("flag")
                )
                if trigger_name:
                    triggers.append({
                        "trigger": str(trigger_name),
                        "meaning": str(item.get("meaning", "")),
                        "expected_answer_behavior": str(item.get("expected_answer_behavior", "")),
                        "validation_status": "draft heuristic; requires guideline or clinician validation",
                    })

    elif isinstance(raw, dict):
        for k, v in raw.items():
            if is_truthy_flag(v):
                if isinstance(v, dict):
                    triggers.append({
                        "trigger": str(k),
                        "meaning": str(v.get("meaning", "")),
                        "expected_answer_behavior": str(v.get("expected_answer_behavior", "")),
                        "validation_status": "draft heuristic; requires guideline or clinician validation",
                    })
                else:
                    triggers.append({
                        "trigger": str(k),
                        "meaning": str(v),
                        "expected_answer_behavior": "",
                        "validation_status": "draft heuristic; requires guideline or clinician validation",
                    })

    return triggers


def get_profile_id(profile: Dict[str, Any]) -> str:
    patient_id = get_first_existing_field(profile, [
        "patient_id",
        "patient_id_norm",
        "inpatient_number",
        "inpatient.number",
        "inpatient number",
        "id",
    ])

    if patient_id is None:
        raise ValueError("A patient profile is missing patient_id or inpatient.number.")

    return normalize_patient_id(patient_id)


def build_fallback_profile_summary(profile: Dict[str, Any]) -> str:
    useful_sections = [
        "demographics",
        "heart_failure_status",
        "hf_status",
        "nyha_class",
        "killip_grade",
        "lvef_category",
        "renal_profile",
        "electrolyte_profile",
        "vitals",
        "comorbidities",
        "medication_flags",
        "medications",
        "final_risk_flags",
        "risk_flags",
    ]

    lines = []

    for key in useful_sections:
        value = get_first_existing_field(profile, [key])
        if value is not None and value != "" and value != [] and value != {}:
            lines.append(f"{key}:\n{stringify_value(value)}")

    if not lines:
        lines.append(stringify_value(profile))

    return "\n\n".join(lines)


def get_patient_context_for_retrieval(profile: Dict[str, Any]) -> str:
    value = get_first_existing_field(profile, [
        "patient_context_for_retrieval",
        "context_for_retrieval",
        "retrieval_context",
        "patient_retrieval_context",
    ])

    if value is None or not str(value).strip():
        value = build_fallback_profile_summary(profile)

    return truncate_text(value, MAX_PATIENT_RETRIEVAL_CONTEXT_CHARS)


def get_patient_context_for_prompt(profile: Dict[str, Any]) -> str:
    value = get_first_existing_field(profile, [
        "patient_context_for_prompt",
        "context_for_prompt",
        "prompt_context",
        "patient_prompt_context",
        "final_rag_profile_summary",
        "final_RAG_profile_summary",
        "rag_profile_summary",
    ])

    if value is None or not str(value).strip():
        value = build_fallback_profile_summary(profile)

    risk_flags = collect_risk_flags(profile)
    triggers = collect_verification_triggers(profile)

    extra_lines = []

    if risk_flags:
        extra_lines.append(
            "Code-derived patient risk flags. These are patient-context signals, not guideline evidence:\n"
            + "\n".join(f"- {x}" for x in risk_flags)
        )

    if triggers:
        extra_lines.append(
            "Code-derived verification triggers. Expected behaviors are draft verification heuristics requiring guideline or clinician validation:\n"
            + "\n".join(f"- {t['trigger']}: {t.get('meaning', '')}" for t in triggers)
        )

    full_context = str(value).strip()
    if extra_lines:
        full_context += "\n\n" + "\n\n".join(extra_lines)

    return truncate_text(full_context, MAX_PATIENT_PROMPT_CONTEXT_CHARS)


with open(FINAL_PATIENT_PROFILE_PATH, "r", encoding="utf-8") as f:
    raw_profile_data = json.load(f)

profile_records = extract_profile_records(raw_profile_data)

PATIENT_PROFILES = {}
for profile in profile_records:
    pid = get_profile_id(profile)
    profile = dict(profile)
    profile["patient_id_norm"] = pid
    PATIENT_PROFILES[pid] = profile

ALL_PATIENT_IDS = sorted(
    PATIENT_PROFILES.keys(),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

print("Loaded patient profiles:", len(PATIENT_PROFILES))
print("First 10 patient IDs:", ALL_PATIENT_IDS[:10])

if len(PATIENT_PROFILES) != 50:
    print("WARNING: Expected 50 final patient profiles, but loaded:", len(PATIENT_PROFILES))


def get_patient_profile(patient_id: Any) -> Dict[str, Any]:
    pid = normalize_patient_id(patient_id)
    if pid not in PATIENT_PROFILES:
        raise ValueError(f"Patient ID not found in final 50-profile JSON: {patient_id}")
    return PATIENT_PROFILES[pid]


def list_patient_ids(limit: int = 20) -> List[str]:
    return ALL_PATIENT_IDS[:limit]

Loaded patient profiles: 50
First 10 patient IDs: ['730098', '730165', '732617', '734179', '738666', '740278', '750142', '750467', '754833', '779822']


cell 6

In [7]:
sample_patient_id = ALL_PATIENT_IDS[0]
sample_profile = get_patient_profile(sample_patient_id)

print("Sample patient ID:", sample_patient_id)
print("\nRisk flags:")
print(json.dumps(collect_risk_flags(sample_profile), indent=2, ensure_ascii=False))

print("\nVerification triggers:")
print(json.dumps(collect_verification_triggers(sample_profile), indent=2, ensure_ascii=False))

print("\nRetrieval context preview:")
print(get_patient_context_for_retrieval(sample_profile)[:1500])

print("\nPrompt context preview:")
print(get_patient_context_for_prompt(sample_profile)[:2500])

Sample patient ID: 730098

Risk flags:
[
  "HFpEF",
  "Killip_II",
  "NYHA_III",
  "ckd_history_or_comorbidity",
  "liver_disease",
  "on_MRA_spironolactone",
  "on_antiplatelet",
  "on_diuretic",
  "on_nitrate",
  "oxygen_therapy_recorded_during_hospitalization",
  "severe_hf_status",
  "severe_renal_risk"
]

Verification triggers:
[]

Retrieval context preview:
{'retrieval_keywords': ['HFpEF', 'Killip II', 'MRA', 'NYHA III', 'anticoagulant', 'antiplatelet', 'bleeding risk', 'diuretic', 'eGFR', 'fluid balance', 'kidney function', 'potassium monitoring', 'red flag symptoms', 'renal function', 'severe_renal_risk', 'spironolactone', 'urgent care', 'weight monitoring', 'worsening heart failure'], 'retrieval_profile_text': 'HFpEF Killip II MRA NYHA III anticoagulant antiplatelet bleeding risk diuretic eGFR fluid balance kidney function potassium monitoring red flag symptoms renal function severe_renal_risk spironolactone urgent care weight monitoring worsening heart failure'}

Prompt conte

cell 7

In [8]:
if not CHILD_NODES_PATH.exists():
    raise FileNotFoundError(f"Missing file: {CHILD_NODES_PATH}")

if not PARENT_LOOKUP_PATH.exists():
    raise FileNotFoundError(f"Missing file: {PARENT_LOOKUP_PATH}")

child_nodes_raw = read_jsonl(CHILD_NODES_PATH)

with open(PARENT_LOOKUP_PATH, "r", encoding="utf-8") as f:
    parent_lookup = json.load(f)

if DOCUMENT_INVENTORY_PATH.exists():
    document_inventory_df = pd.read_csv(DOCUMENT_INVENTORY_PATH)
else:
    document_inventory_df = None

child_nodes_raw = [
    c for c in child_nodes_raw
    if not c.get("metadata", {}).get("index_exclude", False)
]

print("Loaded indexable guideline/document evidence nodes:", len(child_nodes_raw))
print("Loaded parent lookup entries:", len(parent_lookup))

sample_child = child_nodes_raw[0]
print("\nSample child node keys:", sample_child.keys())
print("Sample metadata keys:", sample_child["metadata"].keys())
print("\nSample evidence text preview:")
print(sample_child["text"][:800])

Loaded indexable guideline/document evidence nodes: 1089
Loaded parent lookup entries: 149

Sample child node keys: dict_keys(['node_id', 'parent_id', 'doc_id', 'text', 'metadata'])
Sample metadata keys: dict_keys(['node_id', 'parent_id', 'doc_id', 'source_file', 'source_title', 'publisher', 'year', 'last_updated', 'document_type', 'audience', 'clinical_phase', 'authority_level', 'section_id', 'section_title', 'recommendation_id', 'evidence_role', 'topic', 'page_start', 'page_end', 'parent_strategy', 'child_strategy', 'marker_context', 'is_front_matter', 'index_exclude', 'char_count'])

Sample evidence text preview:
1.2. Organization of the Writing Committee
Guideline-Directed Medical Therapy
The term guideline-directed medical therapy (GDMT) encompasses clinical evaluation, diagnostic testing, and both pharmacological and procedural treatments. For these and all recommended drug treatment regimens, the reader should confirm dosage with product insert material and evaluate for contrain

cell 8

In [9]:
def split_text_semantically_for_index(
    text: str,
    max_chars: int = 2500,
    target_chars: int = 1800,
) -> List[str]:
    text = str(text).strip()

    if len(text) <= max_chars:
        return [text]

    paragraphs = re.split(r"\n\s*\n", text)
    pieces = []

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        if len(para) <= max_chars:
            pieces.append(para)
        else:
            sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", para)
            buf = ""

            for sent in sentences:
                sent = sent.strip()
                if not sent:
                    continue

                if not buf:
                    buf = sent
                elif len(buf) + 1 + len(sent) <= target_chars:
                    buf += " " + sent
                else:
                    if buf:
                        pieces.append(buf)
                    buf = sent

            if buf:
                pieces.append(buf)

    chunks = []
    buf = ""

    for piece in pieces:
        if not buf:
            buf = piece
        elif len(buf) + 2 + len(piece) <= max_chars:
            buf += "\n\n" + piece
        else:
            chunks.append(buf.strip())
            buf = piece

    if buf:
        chunks.append(buf.strip())

    return [c for c in chunks if len(c.strip()) >= 80]


def metadata_to_llama_safe(meta: Dict[str, Any]) -> Dict[str, Any]:
    safe = {}

    for k, v in meta.items():
        if v is None:
            safe[k] = ""
        elif isinstance(v, list):
            safe[k] = " | ".join(str(x) for x in v)
        elif isinstance(v, dict):
            safe[k] = json.dumps(v, ensure_ascii=False)
        elif isinstance(v, (str, int, float, bool)):
            safe[k] = v
        else:
            safe[k] = str(v)

    return safe


def normalize_topic_value(topic_value: Any) -> List[str]:
    if topic_value is None:
        return []

    if isinstance(topic_value, list):
        return [str(x).strip() for x in topic_value if str(x).strip()]

    topic_text = str(topic_value)

    if "|" in topic_text:
        return [x.strip() for x in topic_text.split("|") if x.strip()]

    if "," in topic_text:
        return [x.strip() for x in topic_text.split(",") if x.strip()]

    return [topic_text.strip()] if topic_text.strip() else []

cell 9

In [10]:
import torch

from llama_index.core import Settings, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core.schema import TextNode, MetadataMode, QueryBundle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

try:
    from llama_index.core.llms import MockLLM
    Settings.llm = MockLLM(max_tokens=256)
except Exception:
    Settings.llm = None

print("CUDA available:", torch.cuda.is_available())
print("Embedding device:", EMBED_DEVICE)

embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    device=EMBED_DEVICE,
    embed_batch_size=EMBED_BATCH_SIZE,
)

Settings.embed_model = embed_model

print("LlamaIndex configured.")
print("Embedding model:", EMBED_MODEL_NAME)

CUDA available: True
Embedding device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

LlamaIndex configured.
Embedding model: BAAI/bge-large-en-v1.5


cell 10

In [11]:
def build_llama_nodes_from_child_nodes(child_records: List[Dict[str, Any]]) -> List[TextNode]:
    llama_nodes = []

    for child in tqdm(child_records, desc="Converting guideline evidence chunks to LlamaIndex nodes"):
        text = str(child.get("text", "")).strip()
        if not text:
            continue

        meta = child.get("metadata", {}).copy()
        original_node_id = child.get("node_id") or meta.get("node_id") or stable_hash(text)

        parts = [text]

        if SPLIT_LONG_INDEX_NODES:
            parts = split_text_semantically_for_index(
                text,
                max_chars=MAX_INDEX_NODE_CHARS,
                target_chars=TARGET_INDEX_NODE_CHARS,
            )

        for part_idx, part_text in enumerate(parts, start=1):
            node_id = original_node_id if len(parts) == 1 else f"{original_node_id}_part_{part_idx:02d}"

            part_meta = meta.copy()
            part_meta["node_id"] = node_id
            part_meta["original_node_id"] = original_node_id
            part_meta["index_part"] = part_idx
            part_meta["index_part_count"] = len(parts)
            part_meta["index_text_char_count"] = len(part_text)

            node = TextNode(
                text=part_text,
                id_=node_id,
                metadata=metadata_to_llama_safe(part_meta),
            )

            llama_nodes.append(node)

    return llama_nodes


def index_storage_exists() -> bool:
    if not INDEX_STORAGE_DIR.exists():
        return False

    if not (INDEX_STORAGE_DIR / "index_store.json").exists():
        return False

    has_vector_store = any("vector_store" in p.name for p in INDEX_STORAGE_DIR.iterdir())
    has_docstore = (INDEX_STORAGE_DIR / "docstore.json").exists()

    return has_vector_store and has_docstore


def load_manifest() -> Optional[Dict[str, Any]]:
    if not INDEX_MANIFEST_PATH.exists():
        return None

    with open(INDEX_MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def save_manifest(manifest: Dict[str, Any]) -> None:
    with open(INDEX_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)


def manifest_matches_current_corpus(manifest: Optional[Dict[str, Any]], current_hash: str) -> bool:
    if manifest is None:
        return False

    checks = [
        manifest.get("child_nodes_hash") == current_hash,
        manifest.get("embedding_model") == EMBED_MODEL_NAME,
        manifest.get("split_long_index_nodes") == SPLIT_LONG_INDEX_NODES,
        manifest.get("max_index_node_chars") == MAX_INDEX_NODE_CHARS,
    ]

    return all(checks)


def build_or_load_index():
    current_child_hash = compute_file_hash(CHILD_NODES_PATH)
    manifest = load_manifest()

    can_load_existing = (
        index_storage_exists()
        and manifest_matches_current_corpus(manifest, current_child_hash)
        and not REBUILD_INDEX
    )

    corpus_changed = manifest is not None and manifest.get("child_nodes_hash") != current_child_hash

    if corpus_changed and AUTO_REBUILD_IF_CORPUS_CHANGED:
        can_load_existing = False

    if can_load_existing:
        print("Loading existing persisted LlamaIndex index from Google Drive...")
        storage_context = StorageContext.from_defaults(
            persist_dir=str(INDEX_STORAGE_DIR)
        )
        loaded_index = load_index_from_storage(storage_context)
        print("Index loaded successfully.")
        return loaded_index

    print("Building a new LlamaIndex vector index...")
    print("This may take several minutes the first time.")

    if INDEX_STORAGE_DIR.exists():
        shutil.rmtree(INDEX_STORAGE_DIR)
    INDEX_STORAGE_DIR.mkdir(parents=True, exist_ok=True)

    llama_nodes = build_llama_nodes_from_child_nodes(child_nodes_raw)
    print("Total LlamaIndex nodes to index:", len(llama_nodes))

    new_index = VectorStoreIndex(
        llama_nodes,
        show_progress=True,
    )

    new_index.storage_context.persist(
        persist_dir=str(INDEX_STORAGE_DIR)
    )

    new_manifest = {
        "index_name": "verify_hf_guideline_evidence_index",
        "created_at": datetime.now().isoformat(),
        "embedding_model": EMBED_MODEL_NAME,
        "embedding_device": EMBED_DEVICE,
        "child_nodes_file": str(CHILD_NODES_PATH),
        "child_nodes_hash": current_child_hash,
        "original_child_node_count": len(child_nodes_raw),
        "llama_index_node_count": len(llama_nodes),
        "split_long_index_nodes": SPLIT_LONG_INDEX_NODES,
        "max_index_node_chars": MAX_INDEX_NODE_CHARS,
        "target_index_node_chars": TARGET_INDEX_NODE_CHARS,
        "index_storage_dir": str(INDEX_STORAGE_DIR),
    }

    save_manifest(new_manifest)

    print("Index built and persisted to:", INDEX_STORAGE_DIR)
    print("Manifest saved to:", INDEX_MANIFEST_PATH)

    gc.collect()
    return new_index


index = build_or_load_index()

Loading existing persisted LlamaIndex index from Google Drive...
Index loaded successfully.


cell 11

In [12]:
retriever = index.as_retriever(
    similarity_top_k=RETRIEVE_TOP_K
)

reranker = None


class SentenceTransformerCrossEncoderReranker:
    """
    Stable reranker wrapper using SentenceTransformers CrossEncoder.
    """

    def __init__(
        self,
        model_name: str,
        top_n: int = 10,
        device: str = "cpu",
        batch_size: int = 8,
        max_length: int = 512,
    ):
        from sentence_transformers import CrossEncoder

        self.model_name = model_name
        self.top_n = top_n
        self.device = device
        self.batch_size = batch_size
        self.max_length = max_length

        print(f"Loading CrossEncoder reranker: {model_name}")
        print(f"Reranker device: {device}")

        self.model = CrossEncoder(
            model_name,
            device=device,
            max_length=max_length,
        )

        print("CrossEncoder reranker loaded successfully.")

    def postprocess_nodes(self, nodes, query_bundle=None, query_str=None):
        if not nodes:
            return []

        if query_str is None:
            if query_bundle is not None and hasattr(query_bundle, "query_str"):
                query_str = query_bundle.query_str
            elif query_bundle is not None:
                query_str = str(query_bundle)
            else:
                raise ValueError("Either query_bundle or query_str must be provided.")

        pairs = []

        for nws in nodes:
            evidence_text = nws.node.get_content(metadata_mode=MetadataMode.NONE)
            pairs.append((query_str, evidence_text))

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

        scores = np.asarray(scores)

        if scores.ndim == 2:
            if scores.shape[1] == 1:
                scores = scores[:, 0]
            else:
                scores = scores[:, -1]

        scored_nodes = []

        for nws, score in zip(nodes, scores):
            try:
                nws.score = float(score)
            except Exception:
                pass

            scored_nodes.append((float(score), nws))

        scored_nodes = sorted(scored_nodes, key=lambda x: x[0], reverse=True)

        return [nws for _, nws in scored_nodes[: self.top_n]]


if USE_RERANKER:
    try:
        reranker = SentenceTransformerCrossEncoderReranker(
            model_name=RERANK_MODEL_NAME,
            top_n=RERANK_TOP_N,
            device=RERANK_DEVICE,
            batch_size=RERANK_BATCH_SIZE,
            max_length=RERANK_MAX_LENGTH,
        )
    except Exception as e:
        print("Could not load CrossEncoder reranker.")
        print("Retrieval will continue without reranking.")
        print("Reranker error:", str(e))
        reranker = None
else:
    print("Reranker disabled.")

print("Retriever ready.")
print("Reranker active:", reranker is not None)

Loading CrossEncoder reranker: BAAI/bge-reranker-base
Reranker device: cpu


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

CrossEncoder reranker loaded successfully.
Retriever ready.
Reranker active: True


cell 12

In [13]:
QUERY_TOPIC_KEYWORDS = {
    "post_discharge_follow_up": [
        "discharge", "post-discharge", "post discharge", "follow-up", "follow up",
        "transition", "care plan", "after hospital", "after discharge"
    ],
    "monitoring": [
        "monitor", "monitoring", "watch", "track", "clinical review",
        "blood pressure", "heart rate", "weight", "symptoms", "vitals"
    ],
    "renal_function_electrolytes": [
        "renal", "kidney", "creatinine", "electrolyte", "electrolytes",
        "potassium", "egfr", "eGFR", "urea", "ckd"
    ],
    "sodium_fluid": [
        "sodium", "hyponatremia", "hypernatremia", "fluid", "water intake",
        "fluid restriction", "salt", "low sodium"
    ],
    "potassium_mra": [
        "potassium", "hyperkalemia", "hypokalemia", "spironolactone",
        "mra", "mineralocorticoid"
    ],
    "diuretic_monitoring": [
        "diuretic", "furosemide", "loop diuretic", "congestion",
        "fluid retention", "urine output", "weight"
    ],
    "medication_management": [
        "medication", "medicine", "dose", "ace", "arb", "arni", "mra",
        "beta blocker", "diuretic", "sglt2", "spironolactone"
    ],
    "warning_signs": [
        "warning", "worsening", "shortness of breath", "dyspnea",
        "swelling", "edema", "weight gain", "chest pain", "dizziness",
        "confusion", "cough", "cannot lie flat"
    ],
    "self_management": [
        "daily weight", "sodium", "salt", "diet", "exercise",
        "adherence", "appointment", "lifestyle", "patient explain"
    ],
    "diagnosis_assessment": [
        "diagnosis", "assessment", "echo", "echocardiography", "ecg",
        "chest x-ray", "blood test", "bnp", "nt-probnp"
    ],
}


INTENT_KEYWORDS = {
    "urgent_escalation": [
        "hospital immediately", "go to hospital", "go to the hospital",
        "need hospital", "should go hospital", "should go to hospital",
        "call ambulance", "call 911", "emergency", "emergency department",
        "emergency room", "urgent", "right away", "immediately",
        "life threatening", "life-threatening", "unstable",
        "worsening fast", "acute deterioration"
    ],
    "red_flag_symptoms": [
        "red flag", "red flags", "warning sign", "warning signs",
        "worsening symptoms", "symptoms to watch", "danger signs",
        "shortness of breath", "dyspnea", "chest pain", "confusion",
        "fainting", "dizziness", "cannot lie flat", "swelling", "edema",
        "rapid weight gain"
    ],
    "medication_safety": [
        "medication", "medicine", "drug", "dose", "adjust",
        "increase", "decrease", "titrate", "diuretic", "furosemide",
        "ace inhibitor", "acei", "arb", "arni", "mra",
        "spironolactone", "beta blocker", "beta-blocker", "sglt2",
        "side effect", "renal function", "kidney function", "creatinine",
        "egfr", "electrolyte", "electrolytes", "potassium", "lab", "labs"
    ],
    "fluid_sodium_advice": [
        "fluid", "water intake", "drink water", "sodium", "salt",
        "low sodium", "fluid restriction", "hyponatremia", "dietary sodium"
    ],
    "weight_monitoring": [
        "weight", "daily weight", "weight gain", "weigh", "scale"
    ],
    "post_discharge_monitoring": [
        "post-discharge", "post discharge", "after discharge",
        "discharge monitoring", "monitor after discharge",
        "home monitoring", "monitoring issues", "what should monitor",
        "what to monitor", "nurse monitor", "follow at home"
    ],
    "diet_activity_followup": [
        "diet", "activity", "exercise", "follow-up", "follow up",
        "appointment", "review", "clinical review", "self-care",
        "self care", "self-management", "lifestyle"
    ],
    "patient_education": [
        "explain", "tell patient", "educate", "education",
        "patient understand", "simple language", "patient-friendly",
        "counsel", "teach", "teach-back", "family"
    ],
    "diagnosis_assessment": [
        "diagnosis", "diagnose", "assessment", "evaluate",
        "echo", "echocardiography", "ecg", "ekg", "chest x-ray",
        "xray", "bnp", "nt-probnp", "biomarker", "hfpef",
        "hfref", "ejection fraction"
    ],
}


INTENT_PRIORITY = [
    "urgent_escalation",
    "medication_safety",
    "red_flag_symptoms",
    "fluid_sodium_advice",
    "weight_monitoring",
    "post_discharge_monitoring",
    "diet_activity_followup",
    "patient_education",
    "diagnosis_assessment",
    "general_clinical_question",
]


def contains_phrase(text: str, phrase: str) -> bool:
    """
    Safer phrase matching.
    Fixes the old bug where short substrings like 'er' or 'ed'
    could match inside normal words and falsely trigger urgent intent.
    """
    text = str(text).lower()
    phrase = str(phrase).lower().strip()

    if not phrase:
        return False

    if re.fullmatch(r"[a-z0-9]{1,3}", phrase):
        return re.search(rf"\b{re.escape(phrase)}\b", text) is not None

    return phrase in text


def classify_query_intents(user_question: str) -> Tuple[str, Set[str]]:
    q = str(user_question).lower()
    detected = set()

    for intent, keywords in INTENT_KEYWORDS.items():
        if any(contains_phrase(q, keyword) for keyword in keywords):
            detected.add(intent)

    if not detected:
        detected.add("general_clinical_question")

    for intent in INTENT_PRIORITY:
        if intent in detected:
            return intent, detected

    return "general_clinical_question", detected


def infer_query_topics(text: str) -> Set[str]:
    text_lower = str(text).lower()
    topics = set()

    for topic, keywords in QUERY_TOPIC_KEYWORDS.items():
        if any(contains_phrase(text_lower, k.lower()) for k in keywords):
            topics.add(topic)

    return topics


def profile_to_search_blob(profile: Dict[str, Any]) -> str:
    parts = [
        get_patient_context_for_retrieval(profile),
        "\n".join(collect_risk_flags(profile)),
        "\n".join(t["trigger"] for t in collect_verification_triggers(profile)),
        json.dumps(get_first_existing_field(profile, ["medication_flags"]) or {}, ensure_ascii=False),
        json.dumps(get_first_existing_field(profile, ["medications"]) or {}, ensure_ascii=False),
        json.dumps(get_first_existing_field(profile, ["comorbidities"]) or {}, ensure_ascii=False),
    ]
    return "\n".join(str(x) for x in parts if x)


def infer_patient_risk_topics(profile: Optional[Dict[str, Any]]) -> Set[str]:
    if profile is None:
        return set()

    blob = profile_to_search_blob(profile).lower()
    topics = set()

    if any(x in blob for x in ["renal", "kidney", "ckd", "egfr", "creatinine", "urea"]):
        topics.add("renal_function_electrolytes")

    if any(x in blob for x in ["sodium", "hyponatremia", "hypernatremia"]):
        topics.add("sodium_fluid")

    if any(x in blob for x in ["potassium", "hyperkalemia", "hypokalemia", "spironolactone", "mra"]):
        topics.add("potassium_mra")

    if any(x in blob for x in ["diuretic", "furosemide", "torsemide", "bumetanide"]):
        topics.add("diuretic_monitoring")

    if any(x in blob for x in ["nyha iv", "nyha class iv", "nyha iii", "nyha class iii", "severe", "advanced"]):
        topics.add("warning_signs")
        topics.add("post_discharge_follow_up")

    if any(x in blob for x in ["oxygen", "respiratory", "dyspnea", "shortness of breath"]):
        topics.add("warning_signs")

    if any(x in blob for x in ["blood pressure", "hypotension", "hypertension", "dizziness"]):
        topics.add("monitoring")

    return topics

cell 13

In [14]:
def build_intent_retrieval_focus(primary_intent: str, detected_intents: Set[str]) -> str:
    focus_blocks = []

    if "urgent_escalation" in detected_intents:
        focus_blocks.append("""
Urgent escalation focus:
- when to call a clinician, emergency service, ED, or hospital
- severe or worsening shortness of breath
- chest pain
- confusion, fainting, or altered mental status
- inability to lie flat
- worsening swelling or rapid weight gain
- respiratory distress or oxygenation concern
- worsening heart-failure symptoms after discharge
""".strip())

    if "red_flag_symptoms" in detected_intents:
        focus_blocks.append("""
Red-flag symptom focus:
- warning signs after heart-failure discharge
- worsening shortness of breath or dyspnea
- shortness of breath at rest or lying flat
- sudden or rapid weight gain
- worsening edema or swelling
- chest pain, confusion, fainting, severe dizziness
- when symptoms require urgent clinical assessment
""".strip())

    if "medication_safety" in detected_intents:
        focus_blocks.append("""
Medication safety and lab monitoring focus:
- renal function monitoring
- electrolyte and potassium monitoring
- ACE inhibitor, ARB, ARNI, MRA, beta-blocker, SGLT2 inhibitor, or diuretic monitoring
- medication side effects and intolerance
- clinician review before medication changes
""".strip())

    if "fluid_sodium_advice" in detected_intents:
        focus_blocks.append("""
Fluid and sodium focus:
- sodium restriction or dietary sodium advice
- fluid intake or fluid restriction when relevant
- hyponatremia or sodium abnormality when present in patient context
- diuretic therapy and congestion monitoring
- clinician guidance for individualized fluid advice
""".strip())

    if "weight_monitoring" in detected_intents:
        focus_blocks.append("""
Weight monitoring focus:
- daily weight monitoring
- sudden weight gain thresholds or warning signs
- fluid retention, edema, congestion, and dyspnea
- when weight gain should prompt clinical review
""".strip())

    if "post_discharge_monitoring" in detected_intents:
        focus_blocks.append("""
Post-discharge monitoring focus:
- daily weight, symptoms, blood pressure, heart rate
- dyspnea, edema, congestion, fluid retention
- renal function and electrolytes
- medication adherence and side effects
- follow-up care after discharge
""".strip())

    if "diet_activity_followup" in detected_intents:
        focus_blocks.append("""
Diet, activity, and follow-up focus:
- diet and sodium advice
- physical activity and self-care
- follow-up appointment and care plan
- patient education and monitoring logs
""".strip())

    if "patient_education" in detected_intents:
        focus_blocks.append("""
Patient education focus:
- patient-facing explanations
- self-care instructions
- daily weight checking
- medication adherence
- diet, fluid, sodium, activity, and follow-up
- warning signs explained in simple language
""".strip())

    if "diagnosis_assessment" in detected_intents:
        focus_blocks.append("""
Diagnosis and assessment focus:
- clinical examination and diagnostic testing
- ECG, chest X-ray, blood tests, echocardiography
- BNP or NT-proBNP
- ejection fraction and heart-failure classification
- limitations of available patient context
""".strip())

    if not focus_blocks:
        focus_blocks.append("""
General heart-failure focus:
- post-discharge monitoring
- patient-specific safety considerations
- medication and laboratory monitoring when relevant
- warning signs and follow-up care
""".strip())

    return "\n\n".join(focus_blocks)


def build_patient_aware_query(patient_profile: Dict[str, Any], user_question: str) -> str:
    patient_cues = get_patient_context_for_retrieval(patient_profile)
    risk_flags = collect_risk_flags(patient_profile)
    triggers = collect_verification_triggers(patient_profile)

    primary_intent, detected_intents = classify_query_intents(user_question)
    intent_focus = build_intent_retrieval_focus(primary_intent, detected_intents)

    risk_text = "\n".join(f"- {x}" for x in risk_flags) if risk_flags else "No explicit code-derived risk flags available."
    trigger_text = "\n".join(f"- {t['trigger']}: {t.get('meaning', '')}" for t in triggers) if triggers else "No explicit verification triggers available."

    retrieval_query = f"""
Clinical question from nurse/clinician:
{user_question}

Detected query intent:
Primary intent: {primary_intent}
All detected intents: {", ".join(sorted(detected_intents))}

Patient profile context cues for retrieval.
Important: these are not medical evidence; they only guide retrieval:
{patient_cues}

Code-derived patient risk flags for retrieval prioritization:
{risk_text}

Code-derived verification triggers for retrieval prioritization:
{trigger_text}

Intent-specific retrieval focus:
{intent_focus}

Retrieve guideline/document evidence only. Prefer evidence about:
- post-discharge heart-failure monitoring and follow-up
- patient-specific monitoring priorities
- renal function and electrolyte monitoring when relevant
- sodium, fluid, weight, congestion, dyspnea, edema, blood pressure, and heart rate
- medication safety and clinician review when relevant
- urgent warning signs and escalation needs when relevant
- patient-facing self-care instructions when relevant
""".strip()

    return retrieval_query


def build_standard_retrieval_query(user_question: str) -> str:
    primary_intent, detected_intents = classify_query_intents(user_question)
    intent_focus = build_intent_retrieval_focus(primary_intent, detected_intents)

    return f"""
Clinical question from nurse/clinician:
{user_question}

Detected query intent:
Primary intent: {primary_intent}
All detected intents: {", ".join(sorted(detected_intents))}

No patient profile is used in this baseline.

Intent-specific retrieval focus:
{intent_focus}

Retrieve guideline/document evidence only.
""".strip()

cell 14

In [15]:
def metadata_priority_score(
    meta: Dict[str, Any],
    query_topics: Optional[Set[str]] = None,
    patient_topics: Optional[Set[str]] = None,
) -> float:
    query_topics = query_topics or set()
    patient_topics = patient_topics or set()

    score = 0.0

    document_type = str(meta.get("document_type", "")).lower()
    authority_level = str(meta.get("authority_level", "")).lower()
    evidence_role = str(meta.get("evidence_role", "")).lower()
    source_title = str(meta.get("source_title", "")).lower()
    topics = set(normalize_topic_value(meta.get("topic", "")))
    section_title = str(meta.get("section_title", "")).lower()
    evidence_text_hint = " ".join([
        str(meta.get("topic", "")),
        section_title,
        evidence_role,
        source_title,
    ]).lower()

    if document_type == "clinical_guideline":
        score += 0.20

    if authority_level == "high":
        score += 0.15
    elif authority_level == "supportive":
        score += 0.05

    if "recommendation" in evidence_role:
        score += 0.35
    elif "supportive_text" in evidence_role:
        score += 0.20
    elif "synopsis" in evidence_role:
        score += 0.18
    elif "table" in evidence_role:
        score += 0.15
    elif "warning" in evidence_role or "self_check" in evidence_role:
        score += 0.15
    elif "patient" in evidence_role:
        score += 0.08

    if "aha/acc/hfsa" in source_title:
        score += 0.08

    if "nice" in source_title:
        score += 0.08

    if query_topics and topics.intersection(query_topics):
        score += 0.15

    # Patient-risk-aware topic boost.
    for pt in patient_topics:
        if pt in topics:
            score += 0.20

    # Extra robust keyword-level patient-risk boosts.
    if "renal_function_electrolytes" in patient_topics:
        if any(x in evidence_text_hint for x in ["renal", "kidney", "creatinine", "egfr", "electrolyte"]):
            score += 0.20

    if "sodium_fluid" in patient_topics:
        if any(x in evidence_text_hint for x in ["sodium", "fluid", "salt", "hyponatremia"]):
            score += 0.20

    if "potassium_mra" in patient_topics:
        if any(x in evidence_text_hint for x in ["potassium", "mra", "spironolactone", "aldosterone"]):
            score += 0.20

    if "diuretic_monitoring" in patient_topics:
        if any(x in evidence_text_hint for x in ["diuretic", "furosemide", "congestion", "volume"]):
            score += 0.20

    if "warning_signs" in patient_topics:
        if any(x in evidence_text_hint for x in ["warning", "worsening", "symptom", "dyspnea", "edema"]):
            score += 0.15

    return score


def dedupe_nodes(nodes) -> List[Any]:
    seen = set()
    deduped = []

    for nws in nodes:
        meta = nws.node.metadata
        original_id = meta.get("original_node_id") or meta.get("node_id") or nws.node.node_id
        text = nws.node.get_content(metadata_mode=MetadataMode.NONE)
        key = f"{original_id}_{stable_hash(text[:500])}"

        if key not in seen:
            seen.add(key)
            deduped.append(nws)

    return deduped


def apply_metadata_and_patient_priority(
    nodes,
    retrieval_query: str,
    patient_profile: Optional[Dict[str, Any]] = None,
) -> List[Any]:
    query_topics = infer_query_topics(retrieval_query)
    patient_topics = infer_patient_risk_topics(patient_profile)

    scored = []

    for rank, nws in enumerate(nodes, start=1):
        rank_score = 1.0 / rank
        meta_boost = metadata_priority_score(
            nws.node.metadata,
            query_topics=query_topics,
            patient_topics=patient_topics,
        )

        combined_score = rank_score + meta_boost

        try:
            nws.score = float(combined_score)
        except Exception:
            pass

        scored.append((combined_score, nws))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    return [n for _, n in scored]


def retrieve_evidence(
    retrieval_query: str,
    patient_profile: Optional[Dict[str, Any]] = None,
    final_top_n: int = FINAL_EVIDENCE_TOP_N,
) -> List[Any]:
    """
    Retrieval flow:
    1. Vector retrieval from guideline/document corpus only
    2. Optional CrossEncoder reranking
    3. Deduplication
    4. Metadata + patient-risk-aware priority sorting
    5. Final top-N evidence selection
    """
    initial_nodes = retriever.retrieve(retrieval_query)

    if reranker is not None:
        try:
            reranked_nodes = reranker.postprocess_nodes(
                initial_nodes,
                query_bundle=QueryBundle(query_str=retrieval_query),
            )
        except Exception as e:
            print("Reranker failed; falling back to vector retrieval.")
            print("Reranker error:", str(e))
            reranked_nodes = initial_nodes[:RERANK_TOP_N]
    else:
        reranked_nodes = initial_nodes[:RERANK_TOP_N]

    deduped_nodes = dedupe_nodes(reranked_nodes)
    prioritized_nodes = apply_metadata_and_patient_priority(
        deduped_nodes,
        retrieval_query=retrieval_query,
        patient_profile=patient_profile,
    )

    return prioritized_nodes[:final_top_n]

cell 15

In [16]:
def format_page_range(meta: Dict[str, Any]) -> str:
    page_start = meta.get("page_start", "")
    page_end = meta.get("page_end", "")

    if page_start == page_end or not page_end:
        return str(page_start)

    return f"{page_start}-{page_end}"


def get_parent_excerpt(parent_id: str, max_chars: int = MAX_PARENT_CONTEXT_CHARS) -> str:
    if not parent_id or parent_id not in parent_lookup:
        return ""

    parent_text = parent_lookup[parent_id].get("text", "")
    parent_text = str(parent_text).strip()

    if not parent_text:
        return ""

    return truncate_text(parent_text, max_chars)


def build_evidence_context(nodes, include_parent_context: bool = INCLUDE_PARENT_CONTEXT) -> Tuple[str, List[Dict[str, Any]]]:
    evidence_blocks = []
    references = []

    for i, nws in enumerate(nodes, start=1):
        label = f"S{i}"
        meta = nws.node.metadata
        evidence_text = nws.node.get_content(metadata_mode=MetadataMode.NONE)
        evidence_text = truncate_text(evidence_text, MAX_EVIDENCE_CHARS_PER_SOURCE)

        page_range = format_page_range(meta)
        recommendation_id = meta.get("recommendation_id", "")
        section_title = meta.get("section_title", "")
        source_title = meta.get("source_title", "")
        source_file = meta.get("source_file", "")
        evidence_role = meta.get("evidence_role", "")
        parent_id = meta.get("parent_id", "")

        block = f"""
[{label}]
Source title: {source_title}
Source file: {source_file}
Section: {section_title}
Recommendation ID: {recommendation_id if recommendation_id else "not applicable"}
Page(s): {page_range}
Evidence role: {evidence_role}
Evidence text:
{evidence_text}
""".strip()

        if include_parent_context:
            parent_excerpt = get_parent_excerpt(parent_id)
            if parent_excerpt:
                block += f"\n\nParent section context excerpt:\n{parent_excerpt}"

        evidence_blocks.append(block)

        references.append({
            "label": label,
            "source_title": source_title,
            "source_file": source_file,
            "section_title": section_title,
            "section_id": meta.get("section_id", ""),
            "recommendation_id": recommendation_id,
            "page_range": page_range,
            "evidence_role": evidence_role,
            "authority_level": meta.get("authority_level", ""),
            "document_type": meta.get("document_type", ""),
            "node_id": meta.get("node_id", ""),
            "parent_id": parent_id,
            "score": getattr(nws, "score", None),
            "evidence_text": evidence_text,
        })

    evidence_context = "\n\n---\n\n".join(evidence_blocks)

    return evidence_context, references


def format_references_for_output(references: List[Dict[str, Any]]) -> str:
    lines = ["References used"]

    for ref in references:
        rec_part = ""
        if ref.get("recommendation_id"):
            rec_part = f", recommendation {ref['recommendation_id']}"

        section_part = ""
        if ref.get("section_title"):
            section_part = f" — {ref['section_title']}"

        page_part = ""
        if ref.get("page_range"):
            page_part = f", page(s) {ref['page_range']}"

        role_part = ""
        if ref.get("evidence_role"):
            role_part = f" [{ref['evidence_role']}]"

        line = (
            f"[{ref['label']}] {ref['source_title']}"
            f"{section_part}"
            f"{rec_part}"
            f"{page_part}"
            f"{role_part}."
        )

        lines.append(line)

    return "\n".join(lines)

cell 16

In [17]:
def build_response_format(primary_intent: str) -> str:
    if primary_intent == "urgent_escalation":
        return """
Required response format:
1. Direct answer to the urgent question
2. Patient-specific factors that affect concern
3. Evidence-grounded escalation or monitoring considerations
4. Warning signs that require urgent clinical assessment
5. Suggested clinician/nurse follow-up checks
6. Limitations and uncertainty
""".strip()

    if primary_intent == "medication_safety":
        return """
Required response format:
1. Patient-specific factors relevant to medication safety
2. Evidence-grounded medication or laboratory monitoring considerations
3. Renal/electrolyte or vital-sign cautions when relevant
4. Warning signs or findings needing clinician review
5. Suggested clinician/nurse follow-up questions
6. Limitations and uncertainty
""".strip()

    if primary_intent == "fluid_sodium_advice":
        return """
Required response format:
1. Patient-specific factors relevant to fluid or sodium advice
2. Evidence-grounded fluid/sodium considerations
3. Monitoring issues related to weight, congestion, renal function, or electrolytes
4. When clinician review is needed
5. Limitations and uncertainty
""".strip()

    if primary_intent == "weight_monitoring":
        return """
Required response format:
1. Patient-specific factors relevant to weight monitoring
2. Evidence-grounded daily weight monitoring considerations
3. Warning signs linked to weight gain or fluid retention
4. Suggested clinician/nurse follow-up checks
5. Limitations and uncertainty
""".strip()

    if primary_intent == "red_flag_symptoms":
        return """
Required response format:
1. Patient-specific factors relevant to warning signs
2. Warning signs most relevant to this patient
3. Evidence-grounded escalation considerations
4. What the nurse/clinician should check next
5. Limitations and uncertainty
""".strip()

    if primary_intent == "patient_education":
        return """
Required response format:
1. Patient-specific factors to consider before education
2. Patient-friendly explanation
3. Key self-care instructions
4. Warning signs to explain to the patient
5. Suggested follow-up questions
6. Limitations and uncertainty
""".strip()

    if primary_intent == "diagnosis_assessment":
        return """
Required response format:
1. Patient-specific assessment context
2. Evidence-grounded diagnostic or assessment considerations
3. Available information from the patient context
4. Missing information or checks needed
5. Limitations and uncertainty
""".strip()

    return """
Required response format:
1. Patient-specific monitoring priorities
2. Evidence-grounded monitoring considerations
3. Warning signs to watch
4. Suggested clinician/nurse follow-up questions
5. Limitations and uncertainty
""".strip()


def build_intent_specific_safety_rules(primary_intent: str) -> str:
    if primary_intent == "urgent_escalation":
        return """
Urgent-care question rules:
- Answer the urgent question directly.
- Do not claim that the patient is definitely safe, unstable, or needs admission unless supported by patient context and retrieved evidence.
- If immediate need cannot be determined from retrospective context, say so clearly.
- State that urgent clinical assessment is needed if current red-flag symptoms are present.
- Frame this as clinician/nurse decision support, not direct patient medical advice.
""".strip()

    if primary_intent == "medication_safety":
        return """
Medication safety rules:
- Do not recommend starting, stopping, or changing medication doses unless retrieved evidence directly supports it.
- Focus on monitoring needs, safety checks, and clinician review.
- Do not call lab values high, low, or abnormal unless a threshold or interpretation is available.
""".strip()

    if primary_intent == "fluid_sodium_advice":
        return """
Fluid and sodium rules:
- Do not give individualized fluid restriction as a fact unless retrieved evidence and patient context support it.
- Mention clinician review when renal, sodium, potassium, or diuretic concerns are relevant.
- Do not treat patient-context fields as guideline evidence.
""".strip()

    if primary_intent == "diagnosis_assessment":
        return """
Diagnosis/assessment rules:
- Do not diagnose the patient.
- Do not infer ejection fraction category or HF subtype unless clearly available.
- Separate available patient context from missing clinical information.
""".strip()

    return """
General safety rules:
- Keep the answer focused on clinician/nurse decision support.
- Do not make unsupported treatment changes.
- Do not overstate certainty.
- Do not predict readmission, death, or deterioration as a fact.
""".strip()


def build_patient_specificity_rules() -> str:
    return """
Patient-specific differentiation rules:
- Do not give the same generic answer for every patient.
- Explicitly identify patient-specific factors that change monitoring priority.
- Patient-specific factors may include NYHA class, Killip grade, oxygen use, renal markers, electrolytes, diuretic use, blood pressure, heart rate, comorbidities, BMI/weight, congestion-related features, and cardiac measurements.
- For each monitoring recommendation, connect it to either a patient-specific factor or a retrieved evidence statement.
- If a value is missing or unavailable, say it is unavailable rather than inventing it.
- Do not say “both types of heart failure.” Use safer wording based only on the profile fields.
- Do not call BNP, creatinine, eGFR, sodium, potassium, blood pressure, or other values high/low/abnormal unless threshold or interpretation is available.
""".strip()


def build_citation_rules() -> str:
    return """
Citation rules:
- Use square-bracket citations only: [S1], [S2], [S3].
- Every evidence-grounded clinical recommendation must include at least one citation.
- Patient facts from the patient profile do not require citation.
- Do not cite the patient profile as medical evidence.
- Only cite source IDs that appear in the retrieved evidence blocks.
- Do not invent citations.
- Do not write your own reference list. The system will append a verified reference list.
""".strip()


def build_patient_aware_rag_prompt(
    patient_profile: Dict[str, Any],
    user_question: str,
    evidence_context: str,
) -> str:
    patient_context = get_patient_context_for_prompt(patient_profile)
    primary_intent, detected_intents = classify_query_intents(user_question)

    prompt = f"""
You are a cautious medical AI research assistant for an offline retrospective post-discharge heart-failure follow-up study.

Important setting:
- This is not a real patient consultation.
- This is not a live clinical deployment.
- You support clinician/nurse decision-making in a research prototype.
- You must not replace clinical judgment.
- Do not diagnose the patient.
- Do not make unsupported medication changes.
- Do not mention hidden outcome labels or claim known readmission/death status.

Core evidence rule:
- The patient profile is context only.
- The retrieved guideline/document passages are the only medical evidence source.
- Do not treat patient-profile facts as guideline evidence.
- If retrieved evidence is insufficient, say so clearly.

Detected query intent:
Primary intent: {primary_intent}
All detected intents: {", ".join(sorted(detected_intents))}

Intent-specific safety rules:
{build_intent_specific_safety_rules(primary_intent)}

Patient-specific differentiation rules:
{build_patient_specificity_rules()}

{build_citation_rules()}

Patient profile context:
{patient_context}

Clinician/nurse question:
{user_question}

Retrieved guideline/document evidence:
{evidence_context}

{build_response_format(primary_intent)}

Important final instruction:
The answer must be patient-specific, evidence-grounded, cautious, and citation-aware.

Answer:
""".strip()

    return prompt


def build_standard_rag_prompt(user_question: str, evidence_context: str) -> str:
    primary_intent, detected_intents = classify_query_intents(user_question)

    prompt = f"""
You are a cautious medical AI research assistant for an offline retrospective post-discharge heart-failure follow-up study.

This is the STANDARD RAG baseline:
- No patient profile is provided.
- Use only the retrieved guideline/document evidence.
- Do not invent patient-specific facts.
- Do not diagnose.
- Do not make unsupported medication changes.

Detected query intent:
Primary intent: {primary_intent}
All detected intents: {", ".join(sorted(detected_intents))}

{build_citation_rules()}

Clinician/nurse question:
{user_question}

Retrieved guideline/document evidence:
{evidence_context}

Required response format:
1. General evidence-grounded answer
2. Monitoring or follow-up considerations
3. Warning signs when relevant
4. Limitations and uncertainty

Answer:
""".strip()

    return prompt

cell 17

In [18]:
def prepare_patient_aware_rag(patient_id: Any, question: str) -> Dict[str, Any]:
    profile = get_patient_profile(patient_id)

    retrieval_query = build_patient_aware_query(
        patient_profile=profile,
        user_question=question,
    )

    retrieved_nodes = retrieve_evidence(
        retrieval_query,
        patient_profile=profile,
        final_top_n=FINAL_EVIDENCE_TOP_N,
    )

    evidence_context, references = build_evidence_context(
        retrieved_nodes,
        include_parent_context=INCLUDE_PARENT_CONTEXT,
    )

    prompt = build_patient_aware_rag_prompt(
        patient_profile=profile,
        user_question=question,
        evidence_context=evidence_context,
    )

    return {
        "system_type": "patient_profile_aware_rag",
        "patient_id": normalize_patient_id(patient_id),
        "question": question,
        "patient_profile": profile,
        "patient_context_for_retrieval": get_patient_context_for_retrieval(profile),
        "patient_context_for_prompt": get_patient_context_for_prompt(profile),
        "risk_flags": collect_risk_flags(profile),
        "verification_triggers": collect_verification_triggers(profile),
        "retrieval_query": retrieval_query,
        "retrieved_nodes": retrieved_nodes,
        "evidence_context": evidence_context,
        "references": references,
        "references_text": format_references_for_output(references),
        "prompt": prompt,
    }


def prepare_standard_rag(question: str) -> Dict[str, Any]:
    retrieval_query = build_standard_retrieval_query(question)

    retrieved_nodes = retrieve_evidence(
        retrieval_query,
        patient_profile=None,
        final_top_n=FINAL_EVIDENCE_TOP_N,
    )

    evidence_context, references = build_evidence_context(
        retrieved_nodes,
        include_parent_context=INCLUDE_PARENT_CONTEXT,
    )

    prompt = build_standard_rag_prompt(
        user_question=question,
        evidence_context=evidence_context,
    )

    return {
        "system_type": "standard_rag_question_only",
        "patient_id": None,
        "question": question,
        "retrieval_query": retrieval_query,
        "retrieved_nodes": retrieved_nodes,
        "evidence_context": evidence_context,
        "references": references,
        "references_text": format_references_for_output(references),
        "prompt": prompt,
    }


sample_patient_id = ALL_PATIENT_IDS[0]
sample_question = "What should the nurse monitor after discharge for this heart-failure patient?"

prepared = prepare_patient_aware_rag(sample_patient_id, sample_question)

print("Patient ID:", prepared["patient_id"])
print("Risk flags:", prepared["risk_flags"])
print("\nRetrieval query preview:")
print(prepared["retrieval_query"][:2500])
print("\nReferences:")
print(prepared["references_text"])

Patient ID: 730098
Risk flags: ['HFpEF', 'Killip_II', 'NYHA_III', 'ckd_history_or_comorbidity', 'liver_disease', 'on_MRA_spironolactone', 'on_antiplatelet', 'on_diuretic', 'on_nitrate', 'oxygen_therapy_recorded_during_hospitalization', 'severe_hf_status', 'severe_renal_risk']

Retrieval query preview:
Clinical question from nurse/clinician:
What should the nurse monitor after discharge for this heart-failure patient?

Detected query intent:
Primary intent: post_discharge_monitoring
All detected intents: post_discharge_monitoring

Patient profile context cues for retrieval.
Important: these are not medical evidence; they only guide retrieval:
{'retrieval_keywords': ['HFpEF', 'Killip II', 'MRA', 'NYHA III', 'anticoagulant', 'antiplatelet', 'bleeding risk', 'diuretic', 'eGFR', 'fluid balance', 'kidney function', 'potassium monitoring', 'red flag symptoms', 'renal function', 'severe_renal_risk', 'spironolactone', 'urgent care', 'weight monitoring', 'worsening heart failure'], 'retrieval_pr

cell 18

In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. In Colab, go to Runtime > Change runtime type > GPU."
    )

print("Loading model:", MODEL_ID)

tokenizer_kwargs = {}
if HF_TOKEN:
    tokenizer_kwargs["token"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    **tokenizer_kwargs,
)

model_kwargs = {
    "device_map": "auto",
}

if HF_TOKEN:
    model_kwargs["token"] = HF_TOKEN

if USE_4BIT:
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model_kwargs["quantization_config"] = quantization_config
    model_kwargs["torch_dtype"] = torch.float16

else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **model_kwargs,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

MODEL_INPUT_DEVICE = next(model.parameters()).device

print("Model loaded.")
print("Model input device:", MODEL_INPUT_DEVICE)
print("Tokenizer pad token:", tokenizer.pad_token)
print("USE_4BIT:", USE_4BIT)

Loading model: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded.
Model input device: cuda:0
Tokenizer pad token: </s>
USE_4BIT: True


cell 19

In [26]:
def generate_stream_chunks(user_prompt: str, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    model_inputs = {
        k: v.to(MODEL_INPUT_DEVICE)
        for k, v in model_inputs.items()
    }

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        **model_inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs,
        daemon=True,
    )

    thread.start()

    for new_text in streamer:
        yield new_text


def generate_text(user_prompt: str, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS) -> str:
    chunks = []
    for chunk in generate_stream_chunks(user_prompt, max_new_tokens=max_new_tokens):
        chunks.append(chunk)
    return "".join(chunks).strip()

cell 20

In [28]:
KNOWN_HF_MEDICATION_TERMS = [
    "furosemide", "torsemide", "bumetanide",
    "spironolactone", "eplerenone",
    "enalapril", "lisinopril", "captopril", "ramipril",
    "losartan", "valsartan", "candesartan",
    "sacubitril", "valsartan", "arni",
    "metoprolol", "bisoprolol", "carvedilol",
    "dapagliflozin", "empagliflozin", "sglt2",
    "digoxin", "warfarin",
]


def extract_answer_citation_labels(answer: str) -> Set[str]:
    return set(re.findall(r"\[S\d+\]", str(answer)))


def extract_bare_citation_labels(answer: str) -> Set[str]:
    return set(re.findall(r"(?<!\[)\bS\d+\b(?!\])", str(answer)))


def sentence_split(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+|\n+", str(text))
    return [p.strip() for p in parts if p.strip()]


def looks_like_clinical_recommendation(sentence: str) -> bool:
    s = sentence.lower()

    action_terms = [
        "monitor", "check", "assess", "review", "measure", "track",
        "watch", "follow up", "seek", "call", "refer", "evaluate",
        "avoid", "consider", "should", "need", "requires",
    ]

    clinical_terms = [
        "weight", "blood pressure", "heart rate", "renal", "kidney",
        "creatinine", "egfr", "electrolyte", "potassium", "sodium",
        "fluid", "diuretic", "medication", "symptom", "dyspnea",
        "shortness of breath", "edema", "swelling", "chest pain",
        "follow-up", "follow up", "clinician", "nurse", "doctor",
    ]

    return any(a in s for a in action_terms) and any(c in s for c in clinical_terms)


def validate_citations(answer: str, references: List[Dict[str, Any]]) -> Dict[str, Any]:
    valid_labels = {f"[{ref['label']}]" for ref in references}
    used_labels = extract_answer_citation_labels(answer)
    bare_labels = extract_bare_citation_labels(answer)

    invalid_citations = sorted([x for x in used_labels if x not in valid_labels])

    missing_citation_sentences = []
    for sent in sentence_split(answer):
        if looks_like_clinical_recommendation(sent) and not extract_answer_citation_labels(sent):
            missing_citation_sentences.append(sent)

    status = "pass"
    if invalid_citations:
        status = "fail"
    elif bare_labels or missing_citation_sentences:
        status = "warning"

    return {
        "status": status,
        "valid_labels": sorted(valid_labels),
        "used_labels": sorted(used_labels),
        "invalid_citations": invalid_citations,
        "bare_citation_labels": sorted(bare_labels),
        "missing_citation_sentences": missing_citation_sentences[:10],
    }


def profile_text_blob(profile: Dict[str, Any]) -> str:
    return json.dumps(profile, ensure_ascii=False).lower()


def detect_profile_medications(profile: Dict[str, Any]) -> Set[str]:
    blob = profile_text_blob(profile)
    return {m for m in KNOWN_HF_MEDICATION_TERMS if m in blob}


def find_possible_medication_inventions(answer: str, profile: Dict[str, Any]) -> List[str]:
    """
    Conservative check.
    Flags only when answer appears to say the patient is currently on/taking/receiving a medication
    that is not found anywhere in the structured profile.
    """
    answer_lower = str(answer).lower()
    profile_meds = detect_profile_medications(profile)
    violations = []

    for med in KNOWN_HF_MEDICATION_TERMS:
        if med not in answer_lower:
            continue

        if med in profile_meds:
            continue

        med_positions = [m.start() for m in re.finditer(re.escape(med), answer_lower)]
        for pos in med_positions:
            window = answer_lower[max(0, pos - 80): pos + 80]
            if any(x in window for x in ["patient is on", "patient is taking", "currently on", "receiving", "prescribed"]):
                violations.append(
                    f"Possible invented patient medication: answer implies patient is on '{med}', but this term was not found in the profile."
                )
                break

    return sorted(set(violations))


def find_unavailable_value_claims(answer: str, profile: Dict[str, Any]) -> List[str]:
    """
    Flags cases like: profile says eGFR unavailable, answer says low eGFR.
    """
    profile_blob = profile_text_blob(profile)
    answer_lower = str(answer).lower()
    findings = []

    labs = ["egfr", "lvef", "bnp", "creatinine", "sodium", "potassium", "hemoglobin", "haemoglobin"]

    for lab in labs:
        unavailable_patterns = [
            f"{lab}: not available",
            f"{lab} not available",
            f"{lab}: unavailable",
            f"{lab} unavailable",
            f"{lab}: none",
        ]

        if any(p in profile_blob for p in unavailable_patterns):
            suspicious_patterns = [
                f"low {lab}",
                f"high {lab}",
                f"elevated {lab}",
                f"reduced {lab}",
                f"abnormal {lab}",
            ]

            if any(p in answer_lower for p in suspicious_patterns):
                findings.append(
                    f"Answer interprets {lab} as high/low/elevated/reduced/abnormal although profile indicates it is unavailable."
                )

    return findings


def find_lab_interpretation_warnings(answer: str) -> List[str]:
    """
    Warnings only. This does not prove the answer is wrong.
    It identifies claims that may need threshold/reference-range support.
    """
    answer_lower = str(answer).lower()
    labs = ["egfr", "lvef", "bnp", "creatinine", "sodium", "potassium", "urea", "hemoglobin", "haemoglobin"]
    descriptors = ["high", "low", "elevated", "reduced", "abnormal", "severe"]

    findings = []

    for lab in labs:
        for desc in descriptors:
            pattern1 = f"{desc} {lab}"
            pattern2 = f"{lab} is {desc}"
            pattern3 = f"{lab} was {desc}"

            if pattern1 in answer_lower or pattern2 in answer_lower or pattern3 in answer_lower:
                findings.append(
                    f"Answer interprets {lab} as '{desc}'. Check whether threshold/reference-range support is available."
                )

    return sorted(set(findings))


def patient_profile_consistency_check(answer: str, profile: Dict[str, Any]) -> Dict[str, Any]:
    medication_violations = find_possible_medication_inventions(answer, profile)
    unavailable_value_claims = find_unavailable_value_claims(answer, profile)
    lab_interpretation_warnings = find_lab_interpretation_warnings(answer)

    violations = medication_violations + unavailable_value_claims
    warnings = lab_interpretation_warnings

    status = "pass"
    if violations:
        status = "fail"
    elif warnings:
        status = "warning"

    return {
        "status": status,
        "violations": violations,
        "warnings": warnings,
    }


def red_flag_relevant(question: str, profile: Dict[str, Any]) -> bool:
    primary_intent, detected = classify_query_intents(question)

    if primary_intent in ["urgent_escalation", "red_flag_symptoms"]:
        return True

    trigger_blob = " ".join(t["trigger"] for t in collect_verification_triggers(profile)).lower()
    risk_blob = " ".join(collect_risk_flags(profile)).lower()

    red_terms = ["severe", "nyha_iv", "nyha iv", "oxygen", "dyspnea", "respiratory", "red", "urgent"]

    return any(x in trigger_blob or x in risk_blob for x in red_terms)


def answer_has_escalation_language(answer: str) -> bool:
    a = str(answer).lower()
    terms = [
        "urgent", "emergency", "emergency department", "hospital",
        "call", "clinician", "doctor", "nurse", "medical assessment",
        "clinical assessment", "right away", "immediately", "seek care",
        "seek medical", "same day", "prompt review"
    ]
    return any(t in a for t in terms)


def red_flag_escalation_check(question: str, answer: str, profile: Dict[str, Any]) -> Dict[str, Any]:
    relevant = red_flag_relevant(question, profile)

    if not relevant:
        return {
            "status": "not_applicable",
            "relevant": False,
            "findings": [],
        }

    has_escalation = answer_has_escalation_language(answer)

    if has_escalation:
        return {
            "status": "pass",
            "relevant": True,
            "findings": ["Answer includes clinician/urgent assessment language for red-flag-relevant context."],
        }

    return {
        "status": "fail",
        "relevant": True,
        "findings": ["Red-flag-relevant context detected, but answer lacks clear clinician/urgent assessment language."],
    }


TRIGGER_EXPECTATION_KEYWORDS = {
    "renal": ["renal", "kidney", "creatinine", "egfr", "kidney function", "clinician", "monitor"],
    "ckd": ["renal", "kidney", "creatinine", "egfr", "kidney function", "clinician", "monitor"],
    "potassium": ["potassium", "electrolyte", "mra", "spironolactone", "monitor"],
    "hyperkalemia": ["potassium", "electrolyte", "mra", "spironolactone", "monitor"],
    "hypokalemia": ["potassium", "electrolyte", "diuretic", "monitor"],
    "sodium": ["sodium", "fluid", "hyponatremia", "hypernatremia", "salt", "clinician"],
    "hyponatremia": ["sodium", "fluid", "hyponatremia", "salt", "clinician"],
    "fluid": ["fluid", "weight", "congestion", "edema", "swelling", "dyspnea"],
    "diuretic": ["diuretic", "furosemide", "weight", "fluid", "renal", "electrolyte"],
    "mra": ["mra", "spironolactone", "potassium", "renal", "kidney"],
    "spironolactone": ["spironolactone", "mra", "potassium", "renal", "kidney"],
    "severe": ["urgent", "clinician", "worsening", "warning", "follow-up", "monitor"],
    "nyha": ["nyha", "symptom", "dyspnea", "monitor", "follow-up"],
    "bp": ["blood pressure", "bp", "hypotension", "dizziness", "monitor"],
    "diabetes": ["diabetes", "glucose", "sglt2", "clinician", "monitor"],
}


def trigger_check(answer: str, triggers: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    answer_lower = str(answer).lower()
    results = []

    for trig in triggers:
        trig_name = str(trig.get("trigger", "")).lower()
        matched_policy = None
        required_keywords = []

        for key, keywords in TRIGGER_EXPECTATION_KEYWORDS.items():
            if key in trig_name:
                matched_policy = key
                required_keywords = keywords
                break

        if not matched_policy:
            results.append({
                "trigger": trig.get("trigger", ""),
                "status": "not_checked",
                "meaning": trig.get("meaning", ""),
                "expected_answer_behavior": trig.get("expected_answer_behavior", ""),
                "note": "No deterministic keyword policy defined for this trigger yet.",
                "validation_status": "draft heuristic; requires guideline or clinician validation",
            })
            continue

        passed = any(k in answer_lower for k in required_keywords)

        results.append({
            "trigger": trig.get("trigger", ""),
            "status": "pass" if passed else "warning",
            "meaning": trig.get("meaning", ""),
            "expected_answer_behavior": trig.get("expected_answer_behavior", ""),
            "matched_policy": matched_policy,
            "required_keyword_family": required_keywords,
            "finding": "Answer appears to address this trigger." if passed else "Answer may not clearly address this trigger.",
            "validation_status": "draft heuristic; requires guideline or clinician validation",
        })

    return results


def evidence_support_check(answer: str, references: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    First thesis version: citation-grounded evidence support heuristic.
    This does not prove medical correctness.
    """
    citation_result = validate_citations(answer, references)

    unsupported_claims = citation_result["missing_citation_sentences"]

    status = "pass"
    if unsupported_claims:
        status = "warning"

    return {
        "status": status,
        "method": "heuristic citation-support check; not a clinical correctness proof",
        "unsupported_or_uncited_clinical_claims": unsupported_claims,
    }


def compute_overall_trust_status(
    evidence_support: Dict[str, Any],
    profile_consistency: Dict[str, Any],
    red_flag: Dict[str, Any],
    citation: Dict[str, Any],
    trigger_results: List[Dict[str, Any]],
) -> str:
    if red_flag.get("status") == "fail":
        return "escalation"

    if citation.get("status") == "fail" or profile_consistency.get("status") == "fail":
        return "fail"

    warning_conditions = [
        evidence_support.get("status") == "warning",
        profile_consistency.get("status") == "warning",
        citation.get("status") == "warning",
        any(t.get("status") == "warning" for t in trigger_results),
    ]

    if any(warning_conditions):
        return "warning"

    return "pass"


def verify_answer(
    patient_profile: Dict[str, Any],
    question: str,
    retrieved_evidence: List[Dict[str, Any]],
    answer: str,
    references: List[Dict[str, Any]],
) -> Dict[str, Any]:
    triggers = collect_verification_triggers(patient_profile)

    citation = validate_citations(answer, references)
    profile_consistency = patient_profile_consistency_check(answer, patient_profile)
    red_flag = red_flag_escalation_check(question, answer, patient_profile)
    evidence_support = evidence_support_check(answer, references)
    trigger_results = trigger_check(answer, triggers)

    overall_status = compute_overall_trust_status(
        evidence_support=evidence_support,
        profile_consistency=profile_consistency,
        red_flag=red_flag,
        citation=citation,
        trigger_results=trigger_results,
    )

    return {
        "overall_status": overall_status,
        "important_limitation": (
            "This trust report is a deterministic post-generation verification heuristic. "
            "It does not prove medical correctness and requires guideline and clinician validation."
        ),
        "evidence_support": evidence_support,
        "patient_profile_consistency": profile_consistency,
        "red_flag_escalation": red_flag,
        "citation_correctness": citation,
        "trigger_checks": trigger_results,
        "created_at": datetime.now().isoformat(),
    }

cell 21

In [29]:
def save_json_output(result: Dict[str, Any], prefix: str) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    patient_part = result.get("patient_id") or "no_patient"
    path = RAG_OUTPUT_DIR / f"{prefix}_{patient_part}_{timestamp}.json"

    serializable = dict(result)
    serializable.pop("retrieved_nodes", None)
    serializable.pop("patient_profile", None)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=2, ensure_ascii=False)

    return path


def run_standard_rag(
    question: str,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    save_output: bool = False,
) -> Dict[str, Any]:
    prepared = prepare_standard_rag(question)
    answer = generate_text(prepared["prompt"], max_new_tokens=max_new_tokens)

    final_answer = answer.strip() + "\n\n" + prepared["references_text"]

    result = {
        **prepared,
        "answer": answer,
        "final_answer_with_references": final_answer,
        "model_id": MODEL_ID,
        "embedding_model": EMBED_MODEL_NAME,
        "reranker_model": RERANK_MODEL_NAME if reranker is not None else None,
        "created_at": datetime.now().isoformat(),
    }

    if save_output:
        result["saved_output_path"] = str(save_json_output(result, "standard_rag_output"))

    return result


def run_patient_aware_rag(
    patient_id: Any,
    question: str,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    save_output: bool = False,
) -> Dict[str, Any]:
    prepared = prepare_patient_aware_rag(patient_id, question)
    answer = generate_text(prepared["prompt"], max_new_tokens=max_new_tokens)

    final_answer = answer.strip() + "\n\n" + prepared["references_text"]

    result = {
        **prepared,
        "answer": answer,
        "final_answer_with_references": final_answer,
        "model_id": MODEL_ID,
        "embedding_model": EMBED_MODEL_NAME,
        "reranker_model": RERANK_MODEL_NAME if reranker is not None else None,
        "created_at": datetime.now().isoformat(),
    }

    if save_output:
        result["saved_output_path"] = str(save_json_output(result, "patient_aware_rag_output"))

    return result


def run_patient_aware_rag_with_verification(
    patient_id: Any,
    question: str,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    save_output: bool = False,
) -> Dict[str, Any]:
    result = run_patient_aware_rag(
        patient_id=patient_id,
        question=question,
        max_new_tokens=max_new_tokens,
        save_output=False,
    )

    trust_report = verify_answer(
        patient_profile=result["patient_profile"],
        question=question,
        retrieved_evidence=result["references"],
        answer=result["answer"],
        references=result["references"],
    )

    result["trust_report"] = trust_report

    if save_output:
        result["saved_output_path"] = str(save_json_output(result, "patient_aware_rag_verified_output"))

    return result

cell 22

In [30]:
sample_patient_id = ALL_PATIENT_IDS[0]
sample_question = "What should the nurse monitor after discharge for this heart-failure patient?"

prepared_patient = prepare_patient_aware_rag(sample_patient_id, sample_question)
prepared_standard = prepare_standard_rag(sample_question)

print("PATIENT-AWARE RAG")
print("Patient:", prepared_patient["patient_id"])
print("Risk flags:", prepared_patient["risk_flags"])
print("\nRetrieval query preview:")
print(prepared_patient["retrieval_query"][:2000])
print("\nReferences:")
print(prepared_patient["references_text"])

print("\n" + "="*80 + "\n")

print("STANDARD RAG BASELINE")
print("\nRetrieval query preview:")
print(prepared_standard["retrieval_query"][:1500])
print("\nReferences:")
print(prepared_standard["references_text"])

PATIENT-AWARE RAG
Patient: 730098
Risk flags: ['HFpEF', 'Killip_II', 'NYHA_III', 'ckd_history_or_comorbidity', 'liver_disease', 'on_MRA_spironolactone', 'on_antiplatelet', 'on_diuretic', 'on_nitrate', 'oxygen_therapy_recorded_during_hospitalization', 'severe_hf_status', 'severe_renal_risk']

Retrieval query preview:
Clinical question from nurse/clinician:
What should the nurse monitor after discharge for this heart-failure patient?

Detected query intent:
Primary intent: post_discharge_monitoring
All detected intents: post_discharge_monitoring

Patient profile context cues for retrieval.
Important: these are not medical evidence; they only guide retrieval:
{'retrieval_keywords': ['HFpEF', 'Killip II', 'MRA', 'NYHA III', 'anticoagulant', 'antiplatelet', 'bleeding risk', 'diuretic', 'eGFR', 'fluid balance', 'kidney function', 'potassium monitoring', 'red flag symptoms', 'renal function', 'severe_renal_risk', 'spironolactone', 'urgent care', 'weight monitoring', 'worsening heart failure']

cell 23

In [31]:
sample_patient_id = ALL_PATIENT_IDS[0]
sample_question = "What post-discharge monitoring issues are relevant for this heart-failure patient?"

result = run_patient_aware_rag_with_verification(
    patient_id=sample_patient_id,
    question=sample_question,
    max_new_tokens=600,
    save_output=True,
)

print("Patient ID:", result["patient_id"])
print("\nAnswer:")
print(result["final_answer_with_references"])

print("\nTrust report:")
print(json.dumps(result["trust_report"], indent=2, ensure_ascii=False))

if "saved_output_path" in result:
    print("\nSaved to:", result["saved_output_path"])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Patient ID: 730098

Answer:
1. Patient-specific monitoring priorities:
  - Monitoring for signs of persistent congestion, as the patient is at risk due to NYHA class III and Killip grade II [S1].
  - Close observation of renal function, given the patient's severe renal risk and moderate to severe chronic kidney disease [S1, S2].
  - Regular assessment of electrolyte levels, particularly potassium and sodium, due to the patient's ongoing diuretic use and liver disease [S1, S2, S3].
  - Continued management of blood pressure, given the patient's advanced age and potential for transient reductions in blood pressure [S3].
  - Monitoring for adverse effects of ongoing medications, such as spironolactone, nitrates, and diuretics [S2, S3].

2. Evidence-grounded monitoring considerations:
  - Diuresis should not be discontinued prematurely due to small changes in serum creatinine [S1].
  - Continuation of ACEi-ARB, MRA therapy, and beta blockers after discharge may help prevent recurrent conge

cell 24

In [32]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="VERIFY-HF Patient-Profile-Aware RAG API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


def make_api_safe_result(result: Dict[str, Any]) -> Dict[str, Any]:
    safe = dict(result)
    safe.pop("retrieved_nodes", None)
    safe.pop("patient_profile", None)
    return safe


@app.get("/")
def home():
    return {
        "status": "ok",
        "message": "VERIFY-HF patient-profile-aware RAG API is running",
        "rag_used": True,
        "uses_final_50_patient_profiles": True,
        "patient_profile_role": "context_only_not_medical_evidence",
        "evidence_source": "guideline_document_corpus_only",
        "model_id": MODEL_ID,
        "embedding_model": EMBED_MODEL_NAME,
        "reranker_model": RERANK_MODEL_NAME if reranker is not None else None,
        "total_patients": len(ALL_PATIENT_IDS),
        "available_endpoints": [
            "/patients",
            "/retrieve_patient",
            "/generate_standard",
            "/generate_patient",
            "/generate_patient_verified",
            "/generate_patient_stream",
        ],
    }


@app.get("/patients")
def list_patients_endpoint(limit: int = 20):
    limit = max(1, min(int(limit), 100))
    return {
        "total_patients": len(ALL_PATIENT_IDS),
        "returned": min(limit, len(ALL_PATIENT_IDS)),
        "patient_ids": ALL_PATIENT_IDS[:limit],
    }


@app.post("/retrieve_patient")
async def retrieve_patient_endpoint(request: Request):
    body = await request.json()

    patient_id = body.get("patient_id")
    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    if not patient_id:
        return JSONResponse(status_code=400, content={"error": "patient_id is required"})

    if not question:
        return JSONResponse(status_code=400, content={"error": "question is required"})

    try:
        prepared = prepare_patient_aware_rag(patient_id, question)
        evidence_items = []

        for ref in prepared["references"]:
            evidence_items.append({
                "label": ref["label"],
                "source_title": ref["source_title"],
                "source_file": ref["source_file"],
                "section_title": ref["section_title"],
                "recommendation_id": ref["recommendation_id"],
                "page_range": ref["page_range"],
                "evidence_role": ref["evidence_role"],
                "score": ref.get("score"),
                "evidence_text": truncate_text(ref.get("evidence_text", ""), 1200),
            })

        return {
            "patient_id": prepared["patient_id"],
            "question": question,
            "risk_flags": prepared["risk_flags"],
            "verification_triggers": prepared["verification_triggers"],
            "retrieval_query": prepared["retrieval_query"],
            "evidence_count": len(evidence_items),
            "evidence": evidence_items,
            "references_text": prepared["references_text"],
        }

    except ValueError as e:
        return JSONResponse(status_code=404, content={"error": str(e)})

    except Exception as e:
        print("Retrieval error:", str(e))
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.post("/generate_standard")
async def generate_standard_endpoint(request: Request):
    body = await request.json()

    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    max_new_tokens = int(body.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS))
    save_output = bool(body.get("save_output", False))

    if not question:
        return JSONResponse(status_code=400, content={"error": "question is required"})

    try:
        result = run_standard_rag(
            question=question,
            max_new_tokens=max_new_tokens,
            save_output=save_output,
        )
        return make_api_safe_result(result)

    except Exception as e:
        print("Standard RAG error:", str(e))
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.post("/generate_patient")
async def generate_patient_endpoint(request: Request):
    body = await request.json()

    patient_id = body.get("patient_id")
    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    max_new_tokens = int(body.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS))
    save_output = bool(body.get("save_output", False))

    if not patient_id:
        return JSONResponse(status_code=400, content={"error": "patient_id is required"})

    if not question:
        return JSONResponse(status_code=400, content={"error": "question is required"})

    try:
        result = run_patient_aware_rag(
            patient_id=patient_id,
            question=question,
            max_new_tokens=max_new_tokens,
            save_output=save_output,
        )
        return make_api_safe_result(result)

    except ValueError as e:
        return JSONResponse(status_code=404, content={"error": str(e)})

    except Exception as e:
        print("Patient RAG error:", str(e))
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.post("/generate_patient_verified")
async def generate_patient_verified_endpoint(request: Request):
    body = await request.json()

    patient_id = body.get("patient_id")
    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    max_new_tokens = int(body.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS))
    save_output = bool(body.get("save_output", False))

    if not patient_id:
        return JSONResponse(status_code=400, content={"error": "patient_id is required"})

    if not question:
        return JSONResponse(status_code=400, content={"error": "question is required"})

    try:
        result = run_patient_aware_rag_with_verification(
            patient_id=patient_id,
            question=question,
            max_new_tokens=max_new_tokens,
            save_output=save_output,
        )
        return make_api_safe_result(result)

    except ValueError as e:
        return JSONResponse(status_code=404, content={"error": str(e)})

    except Exception as e:
        print("Verified patient RAG error:", str(e))
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.post("/generate_patient_stream")
async def generate_patient_stream_endpoint(request: Request):
    """
    Streaming endpoint for frontend display.
    This streams the answer and appends references.
    For structured thesis outputs, use /generate_patient_verified.
    """
    body = await request.json()

    patient_id = body.get("patient_id")
    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    max_new_tokens = int(body.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS))

    if not patient_id:
        return JSONResponse(status_code=400, content={"error": "patient_id is required"})

    if not question:
        return JSONResponse(status_code=400, content={"error": "question is required"})

    try:
        prepared = prepare_patient_aware_rag(patient_id, question)
    except Exception as e:
        return JSONResponse(status_code=400, content={"error": str(e)})

    def stream_response():
        full_answer = ""

        try:
            for chunk in generate_stream_chunks(prepared["prompt"], max_new_tokens=max_new_tokens):
                full_answer += chunk
                yield chunk

            ref_text = "\n\n" + prepared["references_text"]
            yield ref_text

        except Exception as e:
            print("Streaming generation error:", str(e))
            yield f"\n[ERROR] {str(e)}"

    return StreamingResponse(
        stream_response(),
        media_type="text/plain",
        headers={"Cache-Control": "no-cache"},
    )

In [40]:
# ============================================================
# Verified streaming endpoint
# Streams answer first, then runs verification, then sends trust report.
# ============================================================

from fastapi.responses import StreamingResponse
import json
from datetime import datetime


def sse_event(event_name: str, data: dict) -> str:
    """
    Format a Server-Sent Event message.
    Frontend will parse these events.
    """
    return (
        f"event: {event_name}\n"
        f"data: {json.dumps(data, ensure_ascii=False)}\n\n"
    )


@app.post("/generate_patient_verified_stream")
async def generate_patient_verified_stream_endpoint(request: Request):
    """
    New endpoint for ChatGPT/Claude-style behavior:

    1. Prepare patient-aware RAG.
    2. Stream answer token-by-token.
    3. Append deterministic references.
    4. Run verification after answer is complete.
    5. Stream trust report as a final structured event.
    """
    body = await request.json()

    patient_id = body.get("patient_id")

    question = (
        body.get("question")
        or body.get("user_question")
        or body.get("prompt")
        or ""
    ).strip()

    max_new_tokens = int(body.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS))
    save_output = bool(body.get("save_output", False))

    if not patient_id:
        return JSONResponse(
            status_code=400,
            content={"error": "patient_id is required"},
        )

    if not question:
        return JSONResponse(
            status_code=400,
            content={"error": "question is required"},
        )

    async def event_generator():
        try:
            # Step 1: Prepare RAG
            yield sse_event(
                "status",
                {
                    "stage": "preparing",
                    "message": "Reading patient profile and preparing patient-aware retrieval.",
                },
            )

            prepared = prepare_patient_aware_rag(
                patient_id=patient_id,
                question=question,
            )

            yield sse_event(
                "retrieval_done",
                {
                    "stage": "retrieval_done",
                    "patient_id": prepared["patient_id"],
                    "risk_flags": prepared.get("risk_flags", []),
                    "verification_triggers": prepared.get("verification_triggers", []),
                    "evidence_count": len(prepared.get("references", [])),
                    "retrieval_query": prepared.get("retrieval_query", ""),
                    "references": prepared.get("references", []),
                    "message": "Guideline/document evidence retrieved and reranked.",
                },
            )

            # Step 2: Stream answer
            yield sse_event(
                "status",
                {
                    "stage": "generating",
                    "message": "Generating cited patient-aware RAG answer.",
                },
            )

            answer_chunks = []

            for chunk in generate_stream_chunks(
                prepared["prompt"],
                max_new_tokens=max_new_tokens,
            ):
                answer_chunks.append(chunk)

                yield sse_event(
                    "answer_delta",
                    {
                        "text": chunk,
                    },
                )

            answer = "".join(answer_chunks).strip()

            # Step 3: Append references
            references_text = "\n\n" + prepared["references_text"]

            yield sse_event(
                "answer_delta",
                {
                    "text": references_text,
                },
            )

            # Step 4: Run verification after full answer exists
            yield sse_event(
                "verification_start",
                {
                    "stage": "verification_start",
                    "message": "Running post-generation verification checks.",
                },
            )

            trust_report = verify_answer(
                patient_profile=prepared["patient_profile"],
                question=question,
                retrieved_evidence=prepared["references"],
                answer=answer,
                references=prepared["references"],
            )

            # Step 5: Save full structured result if requested
            result = {
                **prepared,
                "answer": answer,
                "final_answer_with_references": answer + references_text,
                "trust_report": trust_report,
                "model_id": MODEL_ID,
                "embedding_model": EMBED_MODEL_NAME,
                "reranker_model": RERANK_MODEL_NAME if reranker is not None else None,
                "created_at": datetime.now().isoformat(),
            }

            saved_output_path = None

            if save_output:
                saved_output_path = str(
                    save_json_output(
                        result,
                        "patient_aware_rag_verified_stream_output",
                    )
                )

            # Step 6: Send trust report
            yield sse_event(
                "trust_report",
                {
                    "stage": "verification_done",
                    "message": "Verification complete.",
                    "trust_report": trust_report,
                    "saved_output_path": saved_output_path,
                },
            )

            yield sse_event(
                "done",
                {
                    "stage": "done",
                    "message": "Answer generation and verification complete.",
                },
            )

        except ValueError as e:
            yield sse_event(
                "error",
                {
                    "message": str(e),
                },
            )

        except Exception as e:
            print("Verified streaming endpoint error:", str(e))

            yield sse_event(
                "error",
                {
                    "message": str(e),
                },
            )

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )

cell 25

In [41]:
if RUN_FASTAPI_SERVER:
    import nest_asyncio
    import uvicorn
    from threading import Thread

    nest_asyncio.apply()

    def run_server():
        uvicorn.run(
            app,
            host="0.0.0.0",
            port=8000,
            log_level="info",
        )

    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()

    print("FastAPI server started on port 8000.")
else:
    print("RUN_FASTAPI_SERVER is False.")

FastAPI server started on port 8000.


INFO:     Started server process [668]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


cell 26

In [43]:
# ============================================================
# Start ngrok tunnel
# ============================================================

if START_NGROK:
    from pyngrok import ngrok

    if not NGROK_AUTH_TOKEN or not NGROK_AUTH_TOKEN.strip():
        raise ValueError(
            "NGROK_AUTH_TOKEN is empty. Paste your ngrok token in the configuration cell first."
        )

    if "PASTE_YOUR_NGROK_AUTH_TOKEN_HERE" in NGROK_AUTH_TOKEN:
        raise ValueError(
            "You still have the placeholder ngrok token. Replace it with your real ngrok token."
        )

    ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())

    try:
        ngrok.kill()
    except Exception as e:
        print("No existing ngrok tunnel to kill, or kill failed safely:", str(e))

    tunnel = ngrok.connect(8000, bind_tls=True)
    public_url = tunnel.public_url

    print("Public URL:", public_url)
    print("Patient list endpoint:", public_url + "/patients")
    print("Retrieval endpoint:", public_url + "/retrieve_patient")
    print("Standard RAG endpoint:", public_url + "/generate_standard")
    print("Patient RAG endpoint:", public_url + "/generate_patient")
    print("Verified Patient RAG endpoint:", public_url + "/generate_patient_verified")
    print("Streaming Patient RAG endpoint:", public_url + "/generate_patient_stream")
    print("Verified Streaming Patient RAG endpoint:", public_url + "/generate_patient_verified_stream")

else:
    public_url = "http://localhost:8000"
    print("ngrok disabled.")
    print("Local API URL:", public_url)

Public URL: https://worst-cogwheel-recolor.ngrok-free.dev
Patient list endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/patients
Retrieval endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/retrieve_patient
Standard RAG endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_standard
Patient RAG endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_patient
Verified Patient RAG endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_patient_verified
Streaming Patient RAG endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_patient_stream
Verified Streaming Patient RAG endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_patient_verified_stream


cell 27

In [36]:
import requests

patients_response = requests.get(
    public_url + "/patients",
    params={"limit": 10},
)

print("Status code:", patients_response.status_code)
print(patients_response.json())

sample_patient_id = ALL_PATIENT_IDS[0]

retrieve_response = requests.post(
    public_url + "/retrieve_patient",
    json={
        "patient_id": sample_patient_id,
        "question": "What should the nurse monitor after discharge?",
    },
)

print("\nRetrieve status code:", retrieve_response.status_code)

if retrieve_response.status_code == 200:
    data = retrieve_response.json()
    print("Patient ID:", data["patient_id"])
    print("Evidence count:", data["evidence_count"])
    print("\nRetrieval query preview:")
    print(data["retrieval_query"][:2000])
    print("\nReferences:")
    print(data["references_text"])
else:
    print(retrieve_response.text)

INFO:     127.0.0.1:37844 - "GET /patients?limit=10 HTTP/1.1" 200 OK
Status code: 200
{'total_patients': 50, 'returned': 10, 'patient_ids': ['730098', '730165', '732617', '734179', '738666', '740278', '750142', '750467', '754833', '779822']}
INFO:     127.0.0.1:37858 - "POST /retrieve_patient HTTP/1.1" 200 OK

Retrieve status code: 200
Patient ID: 730098
Evidence count: 8

Retrieval query preview:
Clinical question from nurse/clinician:
What should the nurse monitor after discharge?

Detected query intent:
Primary intent: post_discharge_monitoring
All detected intents: post_discharge_monitoring

Patient profile context cues for retrieval.
Important: these are not medical evidence; they only guide retrieval:
{'retrieval_keywords': ['HFpEF', 'Killip II', 'MRA', 'NYHA III', 'anticoagulant', 'antiplatelet', 'bleeding risk', 'diuretic', 'eGFR', 'fluid balance', 'kidney function', 'potassium monitoring', 'red flag symptoms', 'renal function', 'severe_renal_risk', 'spironolactone', 'urgent ca

cell 28

In [37]:
sample_patient_id = ALL_PATIENT_IDS[0]

verified_response = requests.post(
    public_url + "/generate_patient_verified",
    json={
        "patient_id": sample_patient_id,
        "question": "What post-discharge monitoring issues are relevant for this heart-failure patient?",
        "max_new_tokens": 600,
        "save_output": True,
    },
)

print("Status code:", verified_response.status_code)

if verified_response.status_code == 200:
    data = verified_response.json()

    print("Patient ID:", data["patient_id"])
    print("\nAnswer:")
    print(data["final_answer_with_references"])

    print("\nTrust report:")
    print(json.dumps(data["trust_report"], indent=2, ensure_ascii=False))

    if "saved_output_path" in data:
        print("\nSaved to:", data["saved_output_path"])
else:
    print(verified_response.text)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


INFO:     127.0.0.1:46848 - "POST /generate_patient_verified HTTP/1.1" 200 OK
Status code: 200
Patient ID: 730098

Answer:
1. Patient-specific monitoring priorities:
  - Monitoring for signs of persistent congestion, as the patient is at risk due to NYHA class III and Killip grade II [S1].
  - Close observation of renal function, given the patient's severe renal risk and moderate to severe chronic kidney disease [S1, S2].
  - Regular assessment of electrolyte levels, particularly potassium and sodium, due to the patient's ongoing diuretic use and liver disease [S1, S2, S3].
  - Continued management of blood pressure, given the patient's advanced age and potential for transient reductions in blood pressure [S3].
  - Monitoring for adverse effects of ongoing medications, such as spironolactone, nitrates, and diuretics [S2, S3].

2. Evidence-grounded monitoring considerations:
  - Diuresis should not be discontinued prematurely due to small changes in serum creatinine [S1].
  - Continuati